# Explainable Multilingual Legal Meaning Preservation

## JUDGEBERT-inspired XLM-RoBERTa Regression

This notebook documents the verified outputs of an academic project evaluating whether simplified legal text preserves the meaning of the original in English and Hindi.

**Reproducibility note:** The original experiment notebook was not found in the current Colab runtime or the inspected Drive location. This notebook is a reconstructed analysis notebook for the saved results. It does not claim to reproduce the original training run exactly.

The model is JUDGEBERT-inspired, not an exact reproduction of the original JUDGEBERT implementation.


## 1. Dataset and evaluation setup

The supplied dataset contains 357 sentence pairs in English and Hindi. Its target column is `LMP score`, with ratings from 1 to 10. The dataset's assessment provenance should be verified before describing the ratings as independently validated human annotations.

The reported clean baseline comparison uses a common reconstructed split: 242 training, 43 validation, and 72 test rows. Baseline fitting was restricted to training data. See `results/lmp_split_audit.csv` for the split audit.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/judgebert_lmp_submission")
RESULTS = ROOT / "results"
DATA_PATH = Path("/content/lmp_final - lmp_semantic_review(1).csv")

df = pd.read_csv(DATA_PATH)
df["LMP score"] = pd.to_numeric(df["LMP score"], errors="raise")

print("Dataset shape:", df.shape)
print("Languages:")
display(df["Language"].value_counts().rename_axis("Language").to_frame("Count"))
print("LMP score summary:")
display(df["LMP score"].describe().to_frame("LMP score"))


## 2. Clean baseline comparison

The following metrics are read from the saved comparison CSV. They are descriptive test-set results; no statistical significance testing was performed.


In [ ]:
comparison = pd.read_csv(RESULTS / "lmp_model_comparison_clean.csv")
display(comparison)


## 3. Cross-domain generalization

Two experiments evaluate each model on a domain not used for its training or validation data. Performance is weak in the rental-to-terms/privacy direction.


In [ ]:
held_out = pd.read_csv(RESULTS / "held_out_domain_results.csv")
display(held_out)


## 4. Identical and unrelated sentence sanity checks

The saved checks show that XLM-RoBERTa assigns high scores to some unrelated sentence pairs. This is a major limitation: predicted scores must not be interpreted as proof of legal equivalence.


In [ ]:
sanity = pd.read_csv(RESULTS / "lmp_sanity_check_results.csv")
display(sanity)


## 5. Qualitative legal-error analysis

Fifteen manually reviewed examples cover possible omissions, scope changes, conditions, retention periods, technical details, and terminology changes. These assessments are preliminary and are not legal advice.


In [ ]:
qualitative = pd.read_csv(RESULTS / "lmp_qualitative_case_analysis.csv")
print("Cases:", len(qualitative))
print("Missing values per column:")
display(qualitative.isna().sum().to_frame("Missing"))
display(qualitative)


## 6. Prediction output

The full prediction file contains scores and provisional preservation decisions. The threshold of 8.0 is not independently validated and should be treated as a review heuristic.


In [ ]:
predictions = pd.read_csv(RESULTS / "lmp_prediction_results.csv")
display(predictions.head())
print("Prediction rows:", len(predictions))
print("Decision counts:")
display(predictions["Preservation_Decision"].value_counts().to_frame("Count"))


## 7. Limitations and conclusions

- The dataset is small and its ratings are concentrated toward the high end.
- TF-IDF had the strongest Pearson/Spearman correlations and lowest MAE among the reported clean test baselines; MiniLM had the lowest RMSE.
- XLM-RoBERTa did not outperform the baselines on these metrics.
- The transformer assigned high scores to some unrelated pairs.
- Cross-domain generalization was weak.
- Qualitative review and the threshold-based decision are preliminary.
- Legal meaning preservation requires qualified human review, especially for obligations, exceptions, conditions, dates, amounts, entities, and rights.

This notebook analyzes saved experiment artifacts. It does not independently retrain the models or prove that rerunning training will reproduce the recorded metrics.
